In [1]:
from pathlib import Path

import numpy as np
import json
import pandas as pd
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_data_dir = project_root / "data" / "processed"
raw_data_dir = project_root / "data" / "raw"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"
metrics_dir = project_root / "artifacts" / "metrics"

train_path = processed_data_dir / "train_with_folds.parquet"
test_path = processed_data_dir / "test_clean.parquet"
sample_submission_path = raw_data_dir / "sample_submission.csv"
metrics_dir = project_root / "artifacts" / "metrics"

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)

train_df.shape, test_df.shape, sample_submission_df.shape

((699635, 25), (299844, 23), (299844, 2))

In [3]:
target = train_df["satisfaction"]
fold_numbers = train_df["fold"]
train_ids = train_df["id"]
test_ids = test_df["id"]

features = train_df.drop(columns=["id", "satisfaction", "fold"])
test_features = test_df.drop(columns=["id"])

class_values = sorted(features["Class"].unique())
features["Class"] = pd.Categorical(features["Class"], categories=class_values)
test_features["Class"] = pd.Categorical(test_features["Class"], categories=class_values)

assert list(features.columns) == list(test_features.columns)

features.shape, test_features.shape, features["Class"].dtype

((699635, 22),
 (299844, 22),
 CategoricalDtype(categories=['Business', 'Eco', 'Eco Plus'], ordered=False, categories_dtype=str))

In [15]:
fold = 0

train_mask = fold_numbers != fold
valid_mask = fold_numbers == fold

x_train = features.loc[train_mask]
x_valid = features.loc[valid_mask]

y_train = target.loc[train_mask]
y_valid = target.loc[valid_mask]

x_train.shape, x_valid.shape, y_train.shape, y_valid.shape

((559708, 22), (139927, 22), (559708,), (139927,))

In [16]:
model = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True,
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1,
)

In [17]:
model.fit(
    x_train,
    y_train,
    eval_set = [(x_valid, y_valid)],
    verbose = 100
)

valid_predictions = model.predict_proba(x_valid)[:, 1]

auc = roc_auc_score(y_valid, valid_predictions)

print("Best iteration:", model.best_iteration)
print("Validation ROC AUC:", auc)

Exception ignored on calling ctypes callback function <bound method DataIter._next_wrapper of <xgboost.core.SingleBatchInternalIter object at 0x0000024121829310>>:
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py", line 425, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument
KeyboardInterrupt: 


: 

### Optuna

In [ ]:
import optuna

def objective(trial):
    trial_model = XGBClassifier(
        n_estimators=2000,

        learning_rate=trial.suggest_float(
            "learning_rate",
            0.02,
            0.1,
            log=True
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            4,
            10
        ),

        min_child_weight=trial.suggest_float(
            "min_child_weight",
            1,
            15
        ),

        subsample=trial.suggest_float(
            "subsample",
            0.65,
            1
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            0.65,
            1
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            0.1,
            10,
            log=True
        ),

        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=True,
        early_stopping_rounds=100,
        random_state=42,
        n_jobs=-1,
    )

    trial_model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        verbose=False
    )

    predictions = trial_model.predict_proba(x_valid)[:, 1]

    return roc_auc_score(y_valid, predictions)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [11]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=10,
    show_progress_bar=True
)

print("Best ROC AUC:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-10-05 16:47:54,436] A new study created in memory with name: no-name-00d02bf7-a4c9-4d9f-8fd5-92f2d0eed1c7
Best trial: 0. Best value: 0.958885:  10%|█         | 1/10 [02:16<20:31, 136.79s/it]

[I 2026-10-05 16:50:11,242] Trial 0 finished with value: 0.9588853268741501 and parameters: {'learning_rate': 0.025730289784771848, 'max_depth': 9, 'min_child_weight': 6.700680002076927, 'subsample': 0.7404771554242428, 'colsample_bytree': 0.878504366210371, 'reg_lambda': 0.10786314443309918}. Best is trial 0 with value: 0.9588853268741501.


Best trial: 0. Best value: 0.958885:  20%|██        | 2/10 [04:04<15:57, 119.67s/it]

[I 2026-10-05 16:51:58,942] Trial 1 finished with value: 0.9584093126851463 and parameters: {'learning_rate': 0.06837224675957158, 'max_depth': 7, 'min_child_weight': 3.7020348583783758, 'subsample': 0.6797728913739054, 'colsample_bytree': 0.9156350610782077, 'reg_lambda': 0.17299130985075645}. Best is trial 0 with value: 0.9588853268741501.


Best trial: 0. Best value: 0.958885:  30%|███       | 3/10 [08:25<21:30, 184.35s/it]

[I 2026-10-05 16:56:20,250] Trial 2 finished with value: 0.9588444069566098 and parameters: {'learning_rate': 0.023366638835784062, 'max_depth': 7, 'min_child_weight': 10.99530303682324, 'subsample': 0.8918229423443527, 'colsample_bytree': 0.8697821093911379, 'reg_lambda': 0.6146777206958036}. Best is trial 0 with value: 0.9588853268741501.


... (10 outputs left out)

Best trial: 0. Best value: 0.958885:  90%|█████████ | 9/10 [53:04:41<5:41:23, 20483.27s/it] 

[I 2026-10-07 21:52:35,708] Trial 8 finished with value: 0.9587984463116185 and parameters: {'learning_rate': 0.09642733245839961, 'max_depth': 7, 'min_child_weight': 13.512771581725222, 'subsample': 0.6931855915846945, 'colsample_bytree': 0.7093115250023592, 'reg_lambda': 0.34949006898163404}. Best is trial 0 with value: 0.9588853268741501.


Best trial: 0. Best value: 0.958885: 100%|██████████| 10/10 [53:06:59<00:00, 19121.96s/it] 

[I 2026-10-07 21:54:54,053] Trial 9 finished with value: 0.9587053523455763 and parameters: {'learning_rate': 0.05370316337763673, 'max_depth': 9, 'min_child_weight': 2.5071326862054923, 'subsample': 0.7651423398061136, 'colsample_bytree': 0.8878827899885219, 'reg_lambda': 0.39178510672924877}. Best is trial 0 with value: 0.9588853268741501.
Best ROC AUC: 0.9588853268741501
Best parameters: {'learning_rate': 0.025730289784771848, 'max_depth': 9, 'min_child_weight': 6.700680002076927, 'subsample': 0.7404771554242428, 'colsample_bytree': 0.878504366210371, 'reg_lambda': 0.10786314443309918}


In [3]:
best_params = {
    "learning_rate": 0.025730289784771848,
    "max_depth": 9,
    "min_child_weight": 6.700680002076927,
    "subsample": 0.7404771554242428,
    "colsample_bytree": 0.878504366210371,
    "reg_lambda": 0.10786314443309918,
}

In [8]:
metrics_dir.mkdir(parents=True, exist_ok=True)

params_path = metrics_dir / "xgboost_best_params.json"

with params_path.open("w", encoding="utf-8") as file:
    json.dump(best_params, file, indent=4)

In [9]:
with params_path.open("r", encoding="utf-8") as file:
    saved_params = json.load(file)

saved_params

{'learning_rate': 0.025730289784771848,
 'max_depth': 9,
 'min_child_weight': 6.700680002076927,
 'subsample': 0.7404771554242428,
 'colsample_bytree': 0.878504366210371,
 'reg_lambda': 0.10786314443309918}

In [4]:
import json

params_path = metrics_dir / "xgboost_best_params.json"

with params_path.open("r", encoding="utf-8") as file:
    best_params = json.load(file)

best_params

{'learning_rate': 0.025730289784771848,
 'max_depth': 9,
 'min_child_weight': 6.700680002076927,
 'subsample': 0.7404771554242428,
 'colsample_bytree': 0.878504366210371,
 'reg_lambda': 0.10786314443309918}

In [5]:
fold_scores = []
best_iterations = []

oof_predictions = np.full(len(features), np.nan)
test_predictions = np.zeros(len(test_features))

In [6]:
len(fold_scores)
np.isnan(oof_predictions).sum()
test_predictions.min(), test_predictions.max()

(np.float64(0.0), np.float64(0.0))

In [ ]:
import gc

oof_dir.mkdir(parents=True, exist_ok=True)
test_predictions_dir.mkdir(parents=True, exist_ok=True)

for fold in range(5):
    print(f"\nFold {fold}")

    train_mask = fold_numbers != fold
    valid_mask = fold_numbers == fold

    x_train = features.loc[train_mask]
    x_valid = features.loc[valid_mask]

    y_train = target.loc[train_mask]
    y_valid = target.loc[valid_mask]

    model = XGBClassifier(
        n_estimators=2000,
        learning_rate=best_params["learning_rate"],
        max_depth=best_params["max_depth"],
        min_child_weight=best_params["min_child_weight"],
        subsample=best_params["subsample"],
        colsample_bytree=best_params["colsample_bytree"],
        reg_lambda=best_params["reg_lambda"],
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=True,
        early_stopping_rounds=100,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        verbose=200,
    )

    valid_predictions = model.predict_proba(x_valid)[:, 1]
    fold_test_predictions = model.predict_proba(test_features)[:, 1]

    oof_predictions[valid_mask.to_numpy()] = valid_predictions
    test_predictions += fold_test_predictions / 5

    fold_auc = roc_auc_score(y_valid, valid_predictions)

    fold_scores.append(fold_auc)
    best_iterations.append(model.best_iteration)

    print("ROC AUC:", fold_auc)
    print("Best iteration:", model.best_iteration)

    np.save(
        oof_dir / "xgboost_partial.npy",
        oof_predictions,
    )

    np.save(
        test_predictions_dir / "xgboost_partial.npy",
        test_predictions,
    )

    del model
    del x_train, x_valid
    del y_train, y_valid
    del valid_predictions, fold_test_predictions

    gc.collect()


Fold 0
[0]	validation_0-auc:0.95138
[200]	validation_0-auc:0.95799
[400]	validation_0-auc:0.95882
[551]	validation_0-auc:0.95888
ROC AUC: 0.9588853268741501
Best iteration: 451

Fold 1
[0]	validation_0-auc:0.95084
[200]	validation_0-auc:0.95683
[400]	validation_0-auc:0.95759
[600]	validation_0-auc:0.95771
[773]	validation_0-auc:0.95770
ROC AUC: 0.9577527046591295
Best iteration: 673

Fold 2
[0]	validation_0-auc:0.95213
[200]	validation_0-auc:0.95860
[400]	validation_0-auc:0.95946
[600]	validation_0-auc:0.95965
[800]	validation_0-auc:0.95963
[822]	validation_0-auc:0.95963
ROC AUC: 0.9596857863318297
Best iteration: 722

Fold 3
[0]	validation_0-auc:0.95195
[200]	validation_0-auc:0.95779
[400]	validation_0-auc:0.95855
[600]	validation_0-auc:0.95875
